google collab depedencies

In [50]:
!pip -q install bertopic
!pip -q install sastrawi
!pip -q install gensim

In [51]:
!git clone -q -b gavriel-thesis https://github.com/ranslemus/topic_modeling_KBMI4.git
%cd topic_modeling_KBMI4

/kaggle/working/topic_modeling_KBMI4/topic_modeling_KBMI4


In [52]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px
import random

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from hdbscan.validity import validity_index

# for linux
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN

# for windows
# import umap as UMAP
# import hdbscan as HDBSCAN

In [53]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : Tesla T4


In [54]:
df = pd.read_csv("data/preprocessed_data_downsampled.csv")
df = df[df["year"]==2025]
df.head()

,reviewId,bank,score,year,text
3,629f06db-dc19-4a6b-a526-c5fa09933ed2,LIVIN_MANDIRI_REVIEWS,2,2025,kenapa di login tidak bisa ya malah muncul tul...
10,33535e95-15cb-49b3-bcb7-894957cb159d,WONDR_BNI_REVIEWS,1,2025,ngelag mulu deh
12,8091018d-801d-4a9f-a3ff-86491d781b1e,BRIMO_REVIEWS,1,2025,transaksi berhasil uang enggak masuk gimnaa si...
13,658c217f-74b2-4280-b07a-7ab529fd97a1,BCAMOBILE_REVIEWS,2,2025,sering keluar harus verifikasi lagi terus luma...
17,47ed6779-21fd-45bb-a5a6-c6d92ef93186,BCAMOBILE_REVIEWS,1,2025,malu ih bca mah


In [55]:
df["word_count"] = df["text"].astype(str).str.split().apply(len)
df = df[df["word_count"] >= 5].reset_index(drop=True)
print(f"Total documents setelah filter: {len(df):,}")

Total documents setelah filter: 42,661


In [56]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 42,661


# IndoBERT

In [57]:
MODEL_NAME = "LazarusNLP/simcse-indobert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)

model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: LazarusNLP/simcse-indobert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(50000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [58]:
def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    return torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    ) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

In [59]:
def encode_documents(
    documents,
    batch_size=32,
    max_length=128
):

    embeddings = []

    with torch.no_grad():

        for i in tqdm(
            range(0, len(documents), batch_size)
        ):

            batch = documents[
                i:i+batch_size
            ]

            encoded_input = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            encoded_input = {
                k: v.to(device)
                for k, v in encoded_input.items()
            }

            model_output = model(**encoded_input)

            sentence_embeddings = mean_pooling(
                model_output,
                encoded_input["attention_mask"]
            )

            sentence_embeddings = (
                sentence_embeddings
                .cpu()
                .numpy()
            )

            embeddings.append(sentence_embeddings)

    return np.vstack(embeddings)

In [60]:
embeddings = encode_documents(
    documents,
    batch_size=32,
    max_length=128
)

  0%|          | 0/1334 [00:00<?, ?it/s]

In [61]:
print(embeddings.shape)

(42661, 768)


In [62]:
embeddings[0]

array([ 1.31894362e+00,  9.60366786e-01,  1.30218878e-01, -6.16801903e-02,
        4.10509288e-01, -7.32329428e-01, -1.66571188e+00,  8.28113317e-01,
        8.47559512e-01,  4.62959141e-01, -1.03036702e+00, -1.15735888e+00,
       -9.49390411e-01, -1.97022617e-01, -4.13674146e-01, -5.59255257e-02,
       -8.73682320e-01, -3.98348600e-01,  9.12975729e-01,  1.10419989e-01,
        1.62639177e+00,  6.93946123e-01,  2.03165129e-01, -3.73411298e-01,
       -7.23186851e-01, -1.41948688e+00,  2.93903202e-01, -4.36818421e-01,
       -4.34622020e-01, -1.46654904e-01,  3.04300129e-01,  3.05604726e-01,
        1.74515486e+00,  6.05405420e-02,  2.28680789e-01, -9.99139808e-03,
        6.52468562e-01,  1.10717797e+00, -1.17894602e+00,  1.68272883e-01,
        4.44559038e-01, -1.43568218e-01,  7.49856293e-01, -8.61199439e-01,
       -4.08587456e-01,  2.14764491e-01,  1.07191110e+00,  1.40870261e+00,
        1.30559671e+00,  1.03162718e+00, -1.46915901e+00, -1.55424818e-01,
       -5.29873371e-01,  

In [63]:
norms = np.linalg.norm(embeddings, axis=1)

print("Minimum Norm :", norms.min())
print("Maximum Norm :", norms.max())
print("Average Norm :", norms.mean())
print("Std Norm :", norms.std())

Minimum Norm : 15.018309
Maximum Norm : 26.08848
Average Norm : 21.370525
Std Norm : 1.9197338


In [64]:
print("NaN :", np.isnan(embeddings).sum())
print("Inf :", np.isinf(embeddings).sum())

NaN : 0
Inf : 0


In [65]:
# np.save(
#     "embeddings/indobert_embeddings_downsampled_cutted.npy",
#     embeddings
# )

# BERTopic

In [66]:
# embeddings = np.load("embeddings/indobert_embeddings_downsampled_full.npy")

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (42661, 768)


In [67]:
sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()

# extra_particles = ["banget", "terus", "padahal", "sih", "aja", "saja", "dong", "deh", "ya", "kok", "biar", "gitu", "nih", "loh", "mau", "sudah", "belum"]
sastrawi_stopwords_extended = sastrawi_stopwords 

vectorizer_model = CountVectorizer(
  ngram_range=(1,2),
  stop_words=sastrawi_stopwords,
  token_pattern=r"(?u)\b[^\d\W]+\b",
  min_df=2,
  )

baseline UMAP for testing purpose

In [68]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    metric="cosine",
    min_dist=0.0,
    random_state=42
)

baseline HDBSCAN

In [69]:
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="leaf",
    prediction_data=True
)

In [70]:
from bertopic.vectorizers import ClassTfidfTransformer

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [71]:
topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
)

In [72]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-08-11 06:27:31,724 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-11 06:27:33,447 - BERTopic - Dimensionality - Completed ✓
2026-08-11 06:27:33,449 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-11 06:27:33,793 - BERTopic - Cluster - Completed ✓
2026-08-11 06:27:33,803 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-11 06:27:35,082 - BERTopic - Representation - Completed ✓


# Evaluation

Basic Statistics

In [73]:
topic_info = topic_model.get_topic_info()

topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,29686,-1_bri_uang_brimo_bank,"[bri, uang, brimo, bank, apa, begini, bca, bik...",[aplikasi mobile banking paling jelek sedunia ...
1,0,525,0_verifikasi wajah_wajah_wajah gagal_verifikasi,"[verifikasi wajah, wajah, wajah gagal, verifik...",[kok verifikasi wajah enggak bisa padahal suda...
2,1,463,1_otp_kode otp_kode_otp nya,"[otp, kode otp, kode, otp nya, otp enggak, otp...",[meminta kode otp enggak di kirim kirim ke ema...
3,2,419,2_meminta update_update terus_sering update_up...,"[meminta update, update terus, sering update, ...",[kesel meminta update terus dikit dikit update...
4,3,381,3_bni mobile_mobile_bni_mobile banking,"[bni mobile, mobile, bni, mobile banking, bank...",[sejak pakai wonder sering error enakan mobile...
5,4,375,4_pemeliharaan_maintenance_malam_tiap malam,"[pemeliharaan, maintenance, malam, tiap malam,...",[pliss deh kalau pemeliharaan sistem jangan di...
6,5,365,5_gagal saldo_transaksi gagal_berkurang_saldo ...,"[gagal saldo, transaksi gagal, berkurang, sald...","[transaksi gagal tapi saldo berkurang, transak..."
7,6,329,6_nama_scroll_pencarian_search,"[nama, scroll, pencarian, search, manual, cari...",[setelah update sekarang mau transfer harus ca...
8,7,328,7_foto ktp_foto_ktp_upload,"[foto ktp, foto, ktp, upload, upload foto, ktp...",[kenapa susah sekali upload foto e ktp gagal t...
9,8,300,8_android_oppo_vivo_samsung,"[android, oppo, vivo, samsung, motorola, redmi...",[tidak bisa login sistem operasi anda tidak me...


In [74]:
num_topics = len(topic_info) - 1

outlier_count = (np.array(topics) == -1).sum()

outlier_percentage = (
    outlier_count / len(topics)
) * 100

print(f"Topics              : {num_topics}")
print(f"Outliers            : {outlier_count:,}")
print(f"Outlier Percentage  : {outlier_percentage:.2f}%")

Topics              : 98
Outliers            : 29,686
Outlier Percentage  : 69.59%


Topic Size

In [75]:
topic_info[["Topic","Count"]]

,Topic,Count
0,-1,29686
1,0,525
2,1,463
3,2,419
4,3,381
...,...,...
94,93,52
95,94,51
96,95,50
97,96,50


Top Words

In [76]:
topic_summary = []
for topic in topic_info.Topic:
    if topic == -1:
        continue
    words = ", ".join([w for w, _ in topic_model.get_topic(topic)[:10]])
    topic_summary.append({"Topic": topic, "Count": topic_info.loc[topic_info.Topic==topic, "Count"].values[0], "Top Words": words})

topic_summary_df = pd.DataFrame(topic_summary).sort_values("Count", ascending=False)
topic_summary_df

,Topic,Count,Top Words
0,0,525,"verifikasi wajah, wajah, wajah gagal, verifika..."
1,1,463,"otp, kode otp, kode, otp nya, otp enggak, otp ..."
2,2,419,"meminta update, update terus, sering update, u..."
3,3,381,"bni mobile, mobile, bni, mobile banking, banki..."
4,4,375,"pemeliharaan, maintenance, malam, tiap malam, ..."
...,...,...,...
93,93,52,"karakter, bank bni, al, minjam, meminjam, juta..."
94,94,51,"tanggal waktu, otomatis padahal, otomatis, pen..."
95,95,50,"sangat membantu, membantu, mudah, membantu sek..."
96,96,50,"pin, salah padahal, salah, pin nya, pin benar,..."


Representative Reviews

In [77]:
TOP_N_TOPICS_TO_INSPECT = 15  # cukup buat cek kualitas, nggak perlu semua 30-62 topik

top_topics = topic_info[topic_info.Topic != -1].nlargest(TOP_N_TOPICS_TO_INSPECT, "Count")["Topic"].tolist()
representative_docs = topic_model.get_representative_docs()

for topic in top_topics:
    print(f"\n{'='*80}\nTOPIC {topic} (n={topic_info.loc[topic_info.Topic==topic,'Count'].values[0]})")
    for i, doc in enumerate(representative_docs[topic][:3], 1):
        print(f"{i}. {doc}")


TOPIC 0 (n=525)
1. kok verifikasi wajah enggak bisa padahal sudah cocok sama muka tolong perbaiki lagi dong verifikasi wajahnya sudah enggak bisa login lagi karena verifikasi wajah gagal terus
2. kenapa saya verifikasi wajah gagal terus
3. susah saat verifikasi wajah gagal terus kenapa

TOPIC 1 (n=463)
1. meminta kode otp enggak di kirim kirim ke email
2. kode otp ke gmil enggak masuk masuk
3. sudah kirim kode otp 3x tapi kok enggak ada whatsapp masuk untuk kode otp nya ya

TOPIC 2 (n=419)
1. kesel meminta update terus dikit dikit update dikit dikit update
2. terlalu sering harus di update terus
3. update update terus baru di update kemarin meminta di update lagi bikin memori penuh

TOPIC 3 (n=381)
1. sejak pakai wonder sering error enakan mobile banking yang lama wonder kalo malam suka gangguan kadang saldo tiba-tiba ngurangin harus menunggu beberapa jam muncul lagi saldo normalnya parah harusnya kalo belum siap apk wonder jangan dulu diresmikan mending stay dulu di mobile banking
2.

silhoutte score

In [78]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

silhouette = silhouette_score(
    topic_model.umap_model.embedding_[mask],
    np.array(topics)[mask]
)

print(f"Silhouette Score : {silhouette:.4f}")

Silhouette Score : 0.5618


In [79]:
from itertools import chain

top_n = 10
topic_words = []

for topic in topic_info.Topic:
    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
    ]
    topic_words.append(words)

unique_words = len(
    set(chain.from_iterable(topic_words))
)

total_words = len(topic_words) * top_n
topic_diversity = unique_words / total_words

print(f"Topic Diversity : {topic_diversity:.4f}")

Topic Diversity : 0.8918


Representative Reviews

In [80]:
random.seed(42)

sample_size = 20

for topic_id in sorted(set(topics)):

    if topic_id == -1:
        continue

    topic_docs = [
        doc for doc, topic in zip(documents, topics)
        if topic == topic_id
    ]

    n = min(sample_size, len(topic_docs))
    sampled_docs = random.sample(topic_docs, n)

    print("\n" + "=" * 120)
    print(f"TOPIC {topic_id}")
    print(f"CLUSTER SIZE : {len(topic_docs)}")
    print(f"SAMPLE SIZE  : {n}")
    print("=" * 120)

    for i, doc in enumerate(sampled_docs, 1):
        print(f"{i}. {doc}")


TOPIC 0
CLUSTER SIZE : 525
SAMPLE SIZE  : 20
1. verifikasi wajah susah aplikasi semakin buruk
2. verfikasi wajah nya bagaimana sih susah banget pengin canggih tapi begitu anjir
3. saya tidak dapat melakukan verifikasi wajah
4. terkendala ketika memasuki pengenalan wajah tidak bisa
5. verifikasi wajahnya keulang terus susah lebih bagus apk lama
6. bisa dipermudah lagi enggak untuk verifikasi wajahnya lebih nyaman pakai yang sebelumnya
7. vertifikasi wajah nya susah sudah 4x gagal mulu padahal posisi sudah pas dan sudah mengikuti arahan dengan benar
8. verifikasi wajah selalu gagal padahal sudah di bawah matahari
9. selalu gagal login karena vertikasi wajah bahkan ke kantor cabang pun belum ada solusi
10. sulit verifikasi wajah buat apa ada aplikasi jika sulit di akses
11. kak maaf ini kenapa verifikasi wajah kok tidak bisa mencoba ulang terus
12. verifikasi wajah sangat sulit sudah di lakukan sesuai arahannya tetap saja gagal verifikasi wajah
13. belum saja verifikasi wajah langsung su

NPMI

In [81]:
analyzer = topic_model.vectorizer_model.build_analyzer()

In [82]:
doc.split()

['kenapa',
 'livin',
 'saya',
 'susah',
 'login',
 'selalu',
 'ada',
 'notif',
 'pelayan',
 'tidak',
 'tersedia',
 'untuk',
 'sementara']

In [83]:
tokenized_docs = [
    analyzer(doc)
    for doc in documents
]

In [84]:
from gensim.corpora import Dictionary

dictionary = Dictionary(tokenized_docs)
topic_words = []

for topic in topic_info.Topic:

    if topic == -1:
        continue

    words = []

    for word, score in topic_model.get_topic(topic):
        if word in dictionary.token2id:
            words.append(word)
    # Need at least 2 words for coherence
    if len(words) >= 2:
        topic_words.append(words)

AttributeError: partially initialized module 'smart_open' has no attribute 'local_file' (most likely due to a circular import)

In [ ]:
# sanity check
print(f"Valid Topics : {len(topic_words)}")

print()

print(topic_words[:3])

In [ ]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence="c_npmi"
)

npmi = coherence_model.get_coherence()
print(f"NPMI : {npmi:.4f}")

DBCV

In [ ]:
mask = np.array(topics) != -1
X = topic_model.umap_model.embedding_[mask].astype(np.float64)
labels = np.array(topics)[mask]

dbcv_score = validity_index(X, labels)
print(f"DBCV : {dbcv_score:.4f}")

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

df["topic"] = topics

# 1. Baseline: proporsi tiap bank di keseluruhan korpus
baseline = df["bank"].value_counts(normalize=True) * 100
print("Proporsi bank di keseluruhan korpus (baseline):")
print(baseline.round(2))
print()

# 2. Proporsi tiap bank DI DALAM tiap topik
crosstab = pd.crosstab(df["topic"], df["bank"], normalize="index") * 100
crosstab = crosstab.round(2)

# 3. Hitung "lift" = proporsi di topik / proporsi baseline
#    >1 artinya over-represented di topik itu, <1 artinya under-represented
lift = crosstab.copy()
for bank in baseline.index:
    lift[bank] = crosstab[bank] / baseline[bank]

# 4. Tandai topik yang "njomplang" (deviasi lift > 1.5x atau < 0.5x dari baseline)
def flag_imbalance(row):
    return any(row > 1.5) or any(row < 0.5)

lift["is_imbalanced"] = lift[baseline.index].apply(flag_imbalance, axis=1)

# gabung count per topik biar gampang liat mana yang topik "besar" (bukan cuma noise kecil)
topic_sizes = df[df["topic"] != -1]["topic"].value_counts()
lift["topic_size"] = lift.index.map(topic_sizes)

result = lift[lift.index != -1].sort_values("is_imbalanced", ascending=False)
print(result[list(baseline.index) + ["is_imbalanced", "topic_size"]])

finding the best settings for both HDBSCAN and UMAP

In [ ]:
import itertools

param_grid = {
    "min_cluster_size": [50, 75, 100],
    "min_samples": [5, 10],
    "cluster_selection_epsilon": [0.0, 0.05, 0.1],
}

results = []
combos = list(itertools.product(*param_grid.values()))
print(f"Total kombinasi yang dicoba: {len(combos)}")

for mcs, ms, eps in combos:
    hdbscan_test = HDBSCAN(
        min_cluster_size=mcs,
        min_samples=ms,
        metric="euclidean",
        cluster_selection_method="leaf",
        cluster_selection_epsilon=eps,
        prediction_data=True,
    )
    tm = BERTopic(
        embedding_model=None, calculate_probabilities=False,
        vectorizer_model=vectorizer_model, ctfidf_model=ctfidf_model,
        umap_model=umap_model, hdbscan_model=hdbscan_test, verbose=False,
    )
    tpcs, _ = tm.fit_transform(documents, embeddings)

    ti = tm.get_topic_info()
    n_topics = len(ti) - 1
    outlier_pct = (np.array(tpcs) == -1).sum() / len(tpcs) * 100
    max_share = ti[ti.Topic != -1]["Count"].max() / len(tpcs) * 100 if n_topics > 0 else 0
    mask = np.array(tpcs) != -1
    sil = silhouette_score(tm.umap_model.embedding_[mask], np.array(tpcs)[mask]) if len(set(np.array(tpcs)[mask])) > 1 else float("nan")

    row = {"min_cluster_size": mcs, "min_samples": ms, "epsilon": eps,
           "topics": n_topics, "outlier_%": round(outlier_pct, 2),
           "max_topic_share_%": round(max_share, 2), "silhouette": round(sil, 4)}
    results.append(row)
    print(row)

results_df = pd.DataFrame(results).sort_values("outlier_%")
results_df